# Anansi IA vocal — Colab (gratuit) : Fine-tuning Whisper Wolof + WER

## Ce qu’il te faut
- **Un compte Google** (Gmail) : oui, Colab fonctionne via Google Drive/Google account.
- Un navigateur (Chrome conseillé).
- (Optionnel) un compte Hugging Face si tu veux pousser le modèle, ou si un dataset est privé.

## Comment utiliser ce notebook
1. Ouvre https://colab.research.google.com
2. Connecte-toi avec ton compte Google
3. File → Upload notebook → upload ce fichier `.ipynb`
4. Runtime → Change runtime type → **GPU**
5. Exécute les cellules **dans l’ordre (1 → N)**

> Note: Colab gratuit donne souvent un GPU **T4** (16GB) mais ce n’est pas garanti.

In [ ]:
# Cellule 1 — Vérifier GPU (Colab)
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM (GB):', round(props.total_memory / 1024**3, 2))
else:
    print('⚠️ Pas de GPU détecté. Va dans Runtime → Change runtime type → GPU')

## (Optionnel) Monter Google Drive
Si tu veux sauvegarder les checkpoints et éviter de perdre le modèle quand la session Colab s’arrête.

In [ ]:
# Cellule 2 — Monter Google Drive (optionnel mais recommandé)
from google.colab import drive
drive.mount('/content/drive')

## Charger le code du projet
Deux options: 
- A) **Git clone** si tu as un repo GitHub
- B) **Upload ZIP** si tu as juste le dossier sur ton PC

In [ ]:
# Cellule 3A — (Option A) Cloner le repo (remplace l'URL)
!git clone https://github.com/jnnanna/Anansi-IA-vocal.git  # remplace par ton repo si tu as push
%cd Anansi-IA-vocal  # remplace par le nom du repo cloné
print("Si tu nas pas de repo GitHub, utilise la cellule 3B (upload ZIP).")

In [ ]:
# Cellule 3B — (Option B) Upload un ZIP du projet (recommandé si tu débutes)
from google.colab import files
uploaded = files.upload()
# Upload un fichier ex: Anansi-IA-vocal.zip
import zipfile
import os
zip_name = next(iter(uploaded.keys()))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')
# Trouver le dossier du projet (contient requirements.txt)
proj_root = None
for root, dirs, files_ in os.walk('/content/project'):
    if 'requirements.txt' in files_ and 'src' in dirs:
        proj_root = root
        break
if proj_root is None:
    raise RuntimeError('Impossible de trouver le projet (requirements.txt + src). Vérifie ton ZIP.')
# Change working dir to the project root
os.chdir(proj_root)
print('Projet:', proj_root)

In [ ]:
# Cellule 4 — Installer dépendances
!python -m pip install -U pip setuptools wheel
!pip install -r requirements.txt
# Torch GPU (Colab a souvent déjà torch, mais on garde une commande safe)
# Si ça casse, commente la ligne suivante et garde torch pré-installé par Colab.
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# Cellule 5 — Paramètres dataset/modèle
DATASET_ID = 'KeraCare/Wolof-Kallaama'
MODEL_ID = 'openai/whisper-small'
LANGUAGE = 'wolof'
# Par défaut, on écrit les checkpoints directement sur Google Drive (recommande).
# Si tu ne montes pas Drive, remplace par 'outputs/whisper-small-wo' ou un dossier local.
OUTPUT_DIR = '/content/drive/MyDrive/anansi_wolof_outputs'
print('Dataset:', DATASET_ID)
print('Model:', MODEL_ID)
print('Output:', OUTPUT_DIR)

In [ ]:
# Cellule 6 — Inspecter dataset (colonnes/splits)
!python -m src.inspect_dataset --dataset {DATASET_ID}

In [ ]:
# Cellule 7 — Choisir hyperparams automatiquement selon GPU
import torch
gpu_name = None
vram_gb = None
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    vram_gb = props.total_memory / 1024**3

# Defaults (safe on T4)
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
fp16 = True
bf16 = False
max_steps = 1200  # commence petit; augmente ensuite
learning_rate = 1e-5

if gpu_name and 'A100' in gpu_name:
    per_device_train_batch_size = 16
    gradient_accumulation_steps = 1
    fp16 = False
    bf16 = True
    max_steps = 2000

print('GPU:', gpu_name, 'VRAM(GB):', None if vram_gb is None else round(vram_gb, 1))
print('per_device_train_batch_size =', per_device_train_batch_size)
print('gradient_accumulation_steps =', gradient_accumulation_steps)
print('fp16 =', fp16, 'bf16 =', bf16)
print('max_steps =', max_steps, 'learning_rate =', learning_rate)

In [ ]:
# Cellule 8 — Entraînement Whisper (fine-tuning)
cmd = [
  'python -m src.train_whisper_wolof',
  f'--dataset {DATASET_ID}',
  f'--model {MODEL_ID}',
  f'--language {LANGUAGE}',
  f'--output_dir {OUTPUT_DIR}',
  f'--per_device_train_batch_size {per_device_train_batch_size}',
  f'--gradient_accumulation_steps {gradient_accumulation_steps}',
  f'--learning_rate {learning_rate}',
  f'--max_steps {max_steps}',
]
if fp16:
    cmd.append('--fp16')
if bf16:
    cmd.append('--bf16')
print('\n'.join(cmd))
!{(' '.join(cmd))}

In [ ]:
# Cellule 9 — Évaluer WER sur split validation/test
!python -m src.evaluate_wer --model_dir {OUTPUT_DIR} --dataset {DATASET_ID} --batch_size 8 --max_samples 200

## Sauvegarder le modèle sur Google Drive (recommandé)
Si tu as monté Drive (Cellule 2), copie le dossier `outputs/` vers Drive.

In [ ]:
# Cellule 10 — Copier les outputs vers Google Drive
import os
drive_out = '/content/drive/MyDrive/anansi_wolof_outputs'
os.makedirs(drive_out, exist_ok=True)
!cp -r {OUTPUT_DIR} {drive_out}/
print('Copié vers:', drive_out)

## Test rapide d’inférence sur un fichier audio (.wav) uploadé
Colab n’a pas un accès micro simple et fiable, donc on upload un `wav` depuis ton PC.

In [ ]:
# Cellule 11 — Upload un WAV puis transcrire
from google.colab import files
uploaded = files.upload()
wav_path = next(iter(uploaded.keys()))
print('WAV:', wav_path)

import torch
import soundfile as sf
from transformers import WhisperProcessor, WhisperForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = WhisperProcessor.from_pretrained(OUTPUT_DIR)
model = WhisperForConditionalGeneration.from_pretrained(OUTPUT_DIR).to(device)
model.eval()

audio, sr = sf.read(wav_path)
if audio.ndim > 1:
    audio = audio[:, 0]
inputs = processor(audio, sampling_rate=sr, return_tensors='pt')
input_features = inputs.input_features.to(device)

with torch.no_grad():
    predicted_ids = model.generate(input_features)
text = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
print('Transcription:', text)